# Survivor Data Set: An Exploration
## Project Overview
We seek to understand what it takes to survive in nature, and, more importantly, what it takes to survive in a game-show environment. The "Survivor" TV show is a long-running series, now up to season 50, in which contestants are placed in survival scenarios and compete in various challenges to become the Sole Survivor. The data set explored here accounts for the first 40 of those 50 seasons. The questions we are asking include:
- What does it take to survive a game show about survival?
- What decisions do winners make?
- What decisions do losers make?
- Are there trends that predict performance?
- Are there qualities that predict performance?
- Have there been changes in these trends and qualities over time?

## Data Description and Source
The best data set we found is located here: [Survivor Data Set](https://github.com/rfordatascience/tidytuesday/tree/master/data/2021/2021-06-01)

The GitHub project where this data was found was originally intended for analysis using R; however, this analysis uses pandas.

The data set consists of five CSV files:
- Summary
- Challenges
- Castaways
- Viewers
- Jury Votes



# Initial CSV Data
The following cells provide context for relevance, size, and column descriptions for each CSV file.


In [2]:
# Obligatory pandas import
import pandas as pd

In [3]:
# Each table is available as a CSV from the TidyTuesday GitHub mirror
base_url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-06-01/"

# Load the main tables
summary_df     = pd.read_csv(base_url + "summary.csv")
challenges_df  = pd.read_csv(base_url + "challenges.csv")
castaways_df   = pd.read_csv(base_url + "castaways.csv")
viewers_df     = pd.read_csv(base_url + "viewers.csv")
jury_votes_df  = pd.read_csv(base_url + "jury_votes.csv")

## Summary CSV:

### Relevance:
- Primary data set for information on seasons, winners, and dates.
- Useful for determining winners and trends over time.

### Size:
- 40 rows
- 19 columns
- 6.1 KB

### Column Descriptions (English):
- **Season Name**: The name of the season
- **Season**: The season number
- **Location**: The geographical location of the show
- **Country**: The country where the season takes place
- **Tribe Setup**: How players are divided into tribes (teams)
- **Full Name**: The name of the player
- **Winner**: The winner of the season
- **Runner Ups**: Second place
- **Final Vote**: The vote split for the winner
- **Time Slot**: Day and time of week when episodes aired
- **Premiered**: Date when the season premiered
- **Ended**: Date when the season ended
- **Filming Strated**: Date when filming started (note: this spelling matches the CSV column label)
- **Filming Ended**: Date when filming ended
- **Viewers Finale**: Number of viewers for the final episode
- **Viewers Reunion**: Number of viewers for the reunion (contestant retrospective)
- **Viewers Mean**: Average viewership for episodes
- **Rank**: Viewer final ranking of the contestant

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


In [ ]:
summary_df.info()
summary_df.head()

## Challenges CSV:

### Relevance:
- Examines outcomes when contestants and tribes compete; these outcomes influence decisions when tribes vote contestants out.

### Size:
- 5023 rows
- 8 columns
- 314.1 KB

### Column Descriptions (English):
- **season_name**: The season's name
- **season**: The season number
- **episode**: The episode number within the season
- **title**: The title of the episode
- **day**: The running day count since the game began
- **challenge_type**: Reward (a prize) or immunity (protection from being voted out)
- **winners**: The name of the contestant who won the challenge
- **winning_tribe**: The name of the tribe that won the challenge

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


In [6]:
print(challenges_df.info())
print(challenges_df.head())

# Challenge nulls before cleaning
challenge_nulls_before = challenges_df[["winners", "winning_tribe"]].isna().sum()
print("Nulls before cleaning:")
print(challenge_nulls_before)

<class 'pandas.DataFrame'>
RangeIndex: 5023 entries, 0 to 5022
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   season_name     5023 non-null   str  
 1   season          5023 non-null   int64
 2   episode         5023 non-null   int64
 3   title           5023 non-null   str  
 4   day             5023 non-null   int64
 5   challenge_type  5023 non-null   str  
 6   winners         5006 non-null   str  
 7   winning_tribe   3992 non-null   str  
dtypes: int64(3), str(5)
memory usage: 314.1 KB
None
                season_name  season  episode                   title  day  \
0  Survivor: Winners at War      40        1  Greatest of the Greats    2   
1  Survivor: Winners at War      40        1  Greatest of the Greats    2   
2  Survivor: Winners at War      40        1  Greatest of the Greats    2   
3  Survivor: Winners at War      40        1  Greatest of the Greats    2   
4  Survivor: Winners at War      40  

In [9]:
# Clean Version
challenges_cleaned_df = challenges_df.copy()

# Dropping rows that are placeholder / non-challenge rows
drop_mask = (
    # Survivor: Island of the Idols, episode 12
    ((challenges_cleaned_df["season"] == 39) &
     (challenges_cleaned_df["episode"] == 12) &
     (challenges_cleaned_df["day"] == 36) &
     (challenges_cleaned_df["challenge_type"] == "immunity") &
     (challenges_cleaned_df["winners"].isna())) |

    # Survivor: David vs. Goliath, episode 4
    ((challenges_cleaned_df["season"] == 37) &
     (challenges_cleaned_df["episode"] == 4) &
     (challenges_cleaned_df["day"] == 10) &
     (challenges_cleaned_df["challenge_type"] == "immunity") &
     (challenges_cleaned_df["winners"].isna()))
)

challenges_cleaned_df = challenges_cleaned_df.loc[~drop_mask].copy()

# Filling in rows where there truly was no immunity winner
no_winner_cases = [
    (32, 13),  # Kaoh Rong (Joe evacuated)
    (24, 6),   # One World (Colton evacuated)
    (21, 12),  # Nicaragua (NaOnka and Kelly quit)
    (19, 6),   # Samoa (Russell Swan evacuated)
    (12, 11),  # Panama (Bruce evacuated)
    (8, 3),    # All-Stars (Jenna quit)
    (8, 6),    # All-Stars (Sue quit)
    (2, 6)     # Australian Outback (Michael evacuated)
]

for season_num, episode_num in no_winner_cases:
    mask = (
        (challenges_cleaned_df["season"] == season_num) &
        (challenges_cleaned_df["episode"] == episode_num) &
        (challenges_cleaned_df["challenge_type"] == "immunity") &
        (challenges_cleaned_df["winners"].isna())
    )

    challenges_cleaned_df.loc[mask, "winners"] = "No challenge winner"
    challenges_cleaned_df.loc[mask, "winning_tribe"] = "Not applicable"

# If a winner exists but winning_tribe is missing, then tribe winner does not apply
tribe_not_applicable_mask = (
    challenges_cleaned_df["winners"].notna() &
    challenges_cleaned_df["winning_tribe"].isna()
)

challenges_cleaned_df.loc[tribe_not_applicable_mask, "winning_tribe"] = "Not applicable"

# Fixing Survivor: Blood vs. Water, episode 1
# Galang won the combined immunity/reward challenge, so restore the missing immunity winners
bvw_bad_immunity_rows = (
    (challenges_cleaned_df["season"] == 27) &
    (challenges_cleaned_df["episode"] == 1) &
    (challenges_cleaned_df["day"] == 1) &
    (challenges_cleaned_df["challenge_type"] == "immunity") &
    (challenges_cleaned_df["winners"].isna())
)

challenges_cleaned_df = challenges_cleaned_df.loc[~bvw_bad_immunity_rows].copy()

galang_members = [
    "Aras",
    "Colton",
    "Gervase",
    "Kat",
    "Laura B.",
    "Laura M.",
    "Monica",
    "Tina",
    "Tyson"
]

bvw_immunity_rows = pd.DataFrame(
    [
        {
            "season_name": "Survivor: Blood vs. Water",
            "season": 27,
            "episode": 1,
            "title": "Blood Is Thicker Than Anything",
            "day": 1,
            "challenge_type": "immunity",
            "winners": member,
            "winning_tribe": "Galang"
        }
        for member in galang_members
    ]
)

challenges_cleaned_df = pd.concat(
    [challenges_cleaned_df, bvw_immunity_rows],
    ignore_index=True
)

# Drop the last 3 blank reward placeholder rows
final_placeholder_rows = (
    ((challenges_cleaned_df["season"] == 22) &
     (challenges_cleaned_df["episode"] == 14) &
     (challenges_cleaned_df["day"] == 38) &
     (challenges_cleaned_df["challenge_type"] == "reward") &
     (challenges_cleaned_df["winners"].isna())) |

    ((challenges_cleaned_df["season"] == 16) &
     (challenges_cleaned_df["episode"] == 1) &
     (challenges_cleaned_df["day"] == 3) &
     (challenges_cleaned_df["challenge_type"] == "reward") &
     (challenges_cleaned_df["winners"].isna())) |

    ((challenges_cleaned_df["season"] == 13) &
     (challenges_cleaned_df["episode"] == 6) &
     (challenges_cleaned_df["day"] == 15) &
     (challenges_cleaned_df["challenge_type"] == "reward") &
     (challenges_cleaned_df["winners"].isna()))
)

challenges_cleaned_df = challenges_cleaned_df.loc[~final_placeholder_rows].copy()

# Sort the cleaned df so the column names make sense
challenges_cleaned_df = challenges_cleaned_df.sort_values(
    by=["season", "episode", "day", "challenge_type", "winners"]
).reset_index(drop=True)

# Challenge nulls after cleaning
challenge_nulls_after = challenges_cleaned_df[["winners", "winning_tribe"]].isna().sum()
print("\nChallenge nulls after cleaning:")
print(challenge_nulls_after)

print(challenges_cleaned_df.info())
print(challenges_cleaned_df.head())


Challenge nulls after cleaning:
winners          0
winning_tribe    0
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 5024 entries, 0 to 5023
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   season_name     5024 non-null   str  
 1   season          5024 non-null   int64
 2   episode         5024 non-null   int64
 3   title           5024 non-null   str  
 4   day             5024 non-null   int64
 5   challenge_type  5024 non-null   str  
 6   winners         5024 non-null   str  
 7   winning_tribe   5024 non-null   str  
dtypes: int64(3), str(5)
memory usage: 314.1 KB
None
        season_name  season  episode          title  day challenge_type  \
0  Survivor: Borneo       1        1  The Marooning    3       immunity   
1  Survivor: Borneo       1        1  The Marooning    3       immunity   
2  Survivor: Borneo       1        1  The Marooning    3       immunity   
3  Survivor: Borneo       1        1  

## Castaways CSV:

### Relevance:
- Personal data on each contestant and their performance.
- Useful for analyzing how traits relate to outcomes.

### Size:
- 744 rows
- 18 columns
- 104.8 KB

### Column Descriptions (English):
- **season_name**: The name of the season
- **season**: The season number
- **full_name**: The contestant's full name
- **castaway**: The castaway's (contestant's) first name
- **age**: The contestant's age
- **city**: The city the contestant is from
- **state**: The state the contestant is from
- **personality_type**: Their personality description
- **day**: The day of the season (running count)
- **order**: Finish order for the season (larger values mean the contestant lasted longer)
- **result**: When they were voted out (string description of the order column)
- **jury_status**: If and when the player made the jury
- **original_tribe**: The tribe they started on
- **swapped_tribe**: The tribe they swapped to
- **swapped_tribe2**: The tribe they swapped to a second time
- **merged_tribe**: The merged tribe name
- **total_votes_received**: Number of votes cast against the contestant
- **immunity_idols_won**: Number of immunity idols won by the contestant

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


In [ ]:
castaways_df.info()
castaways_df.head()

## Viewers CSV:

### Relevance:
- Useful for gauging interest in Survivor.
- May reveal relationships between public interest and contestant performance.

### Size:
- 596 rows
- 9 columns
- 42 KB

### Column Descriptions (English):
- **season_name**: The name of the season
- **season**: The season number
- **episode_number_overall**: The episode number across all seasons
- **episode**: The episode number within this season
- **title**: The title of the episode
- **episode_date**: The date the episode aired
- **viewers**: The number of viewers, in millions
- **rating_18_49**: Percentage of TV households in the 18-49 demographic that watched Survivor
- **share_18_49**: Among 18-49 viewers watching TV during the time slot, the percentage who watched Survivor

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


In [ ]:
viewers_df.info()
viewers_df.head()

## Jury Votes CSV:

### Relevance:
- Captures how jury members voted at Final Tribal Council.
- May reveal how personalities affect outcomes in the end.

### Size:
- 909 rows
- 5 columns
- 35.6 KB

### Column Descriptions (English):
- **season_name**: The name of the season
- **season**: The season number
- **castaway**: The juror casting votes
- **finalist**: A finalist for the season who can receive jury votes
- **vote**: Whether the juror voted for this finalist (1 = yes, 0 = no)

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


In [ ]:
jury_votes_df.info()
jury_votes_df.head()